# Setup

In [1]:
# Tìm thư mục gốc của dự án để notebook biết nơi chứa mã nguồn và dữ liệu.
from pathlib import Path
import sys
import pandas as pd

def find_project_root(max_up=6):
    p = Path.cwd()
    for _ in range(max_up + 1):
        if (p / "src").exists() and (p / "configs").exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = find_project_root()

sys.path.append(str(PROJECT_ROOT))

from src.io_utils import save_csv
from src.evidence import create_alert_evidence_package

alerts_df = pd.read_csv(PROJECT_ROOT / "outputs/alerts/alerts.csv")
global_df = pd.read_csv(PROJECT_ROOT / "outputs/global_ids/global_ids.csv")
rep_crops_df = pd.read_csv(PROJECT_ROOT / "outputs/crops/representative_crops.csv")

alerts_df.head()


,global_id,entry_time_sec,time_since_entry_sec,ticket_evidence,journey_state,suspicion_score,alert_level,status,reason
0,GID_00000,9.385362,990.614638,True,ticket_evidence_found,0.677053,medium,watchlist,Entered area detected; Exceeded maximum travel...
1,GID_00001,69.587790,930.412210,False,suspected_no_ticket_evidence,0.960000,high,alert_generated,Entered area detected; No ticket counter evide...
2,GID_00002,95.922704,904.077296,False,suspected_no_ticket_evidence,0.960000,high,alert_generated,Entered area detected; No ticket counter evide...
3,GID_00003,114.181555,885.818445,True,ticket_evidence_found,0.677599,medium,watchlist,Entered area detected; Exceeded maximum travel...
4,GID_00004,117.574053,882.425947,True,ticket_evidence_found,0.642776,medium,watchlist,Entered area detected; Exceeded maximum travel...


# Tạo evidence package cho alert

In [2]:
# Chuẩn bị ảnh và dữ liệu evidence để con người kiểm tra các cảnh báo quan trọng.
evidence_root = PROJECT_ROOT / "outputs/evidence"
evidence_root.mkdir(parents=True, exist_ok=True)

high_alerts = alerts_df[alerts_df["status"] == "alert_generated"].copy()

evidence_rows = []

for _, alert in high_alerts.iterrows():
    gid = alert["global_id"]

    gid_info = global_df[global_df["global_id"] == gid]

    if len(gid_info) == 0:
        continue

    entry_tracklet_id = int(gid_info.iloc[0]["entry_tracklet_id"])

    crop_candidates = rep_crops_df[
        rep_crops_df["local_track_id"] == entry_tracklet_id
    ].sort_values("quality_score", ascending=False)

    crop_paths = crop_candidates["crop_path"].head(3).tolist()

    gid_dir = create_alert_evidence_package(
        global_id=gid,
        alert_row=alert,
        crop_paths=crop_paths,
        evidence_root=evidence_root
    )

    evidence_rows.append({
        "global_id": gid,
        "evidence_folder": str(gid_dir),
        "num_crops": len(crop_paths)
    })

evidence_df = pd.DataFrame(evidence_rows)

out_path = PROJECT_ROOT / "outputs/evidence/evidence_index.csv"
save_csv(evidence_df, out_path)

print("Saved:", out_path)
evidence_df.head()


Saved: D:\hoiancomputervision\ticket_suspicion_project\outputs\evidence\evidence_index.csv


,global_id,evidence_folder,num_crops
0,GID_00001,D:\hoiancomputervision\ticket_suspicion_projec...,3
1,GID_00002,D:\hoiancomputervision\ticket_suspicion_projec...,3
2,GID_00007,D:\hoiancomputervision\ticket_suspicion_projec...,3
3,GID_00008,D:\hoiancomputervision\ticket_suspicion_projec...,3
4,GID_00009,D:\hoiancomputervision\ticket_suspicion_projec...,3


# Tạo human review CSV

In [3]:
# Chuẩn bị ảnh và dữ liệu evidence để con người kiểm tra các cảnh báo quan trọng.
human_review_rows = []

for _, alert in high_alerts.iterrows():
    gid = alert["global_id"]

    evidence_folder = str(evidence_root / gid)

    human_review_rows.append({
        "global_id": gid,
        "suspicion_score": float(alert["suspicion_score"]),
        "alert_level": alert["alert_level"],
        "system_reason": alert["reason"],
        "evidence_folder": evidence_folder,
        "human_decision": "pending",
        "human_note": ""
    })

human_review_df = pd.DataFrame(human_review_rows)

out_path = PROJECT_ROOT / "outputs/human_review.csv"
save_csv(human_review_df, out_path)

print("Saved:", out_path)
human_review_df.head()


Saved: D:\hoiancomputervision\ticket_suspicion_project\outputs\human_review.csv


,global_id,suspicion_score,alert_level,system_reason,evidence_folder,human_decision,human_note
0,GID_00001,0.96,high,Entered area detected; No ticket counter evide...,D:\hoiancomputervision\ticket_suspicion_projec...,pending,
1,GID_00002,0.96,high,Entered area detected; No ticket counter evide...,D:\hoiancomputervision\ticket_suspicion_projec...,pending,
2,GID_00007,0.96,high,Entered area detected; No ticket counter evide...,D:\hoiancomputervision\ticket_suspicion_projec...,pending,
3,GID_00008,0.96,high,Entered area detected; No ticket counter evide...,D:\hoiancomputervision\ticket_suspicion_projec...,pending,
4,GID_00009,0.96,high,Entered area detected; No ticket counter evide...,D:\hoiancomputervision\ticket_suspicion_projec...,pending,
